# World Cup Experience

## Objective

Add 2026 World Cup experience features to each qualified team. This follows the same leakage-safe methods as the manager and player experience notebooks: only World Cup appearances before the 2026 tournament count as prior experience.

The final dataset keeps the 2026 distance and ELO features, then adds manager previous World Cups coached, player multiple previous World Cups played, and average squad age.

## Inputs

- `worldcup::manager_appearances`
- `worldcup::player_appearances`
- `worldcup::squads`
- `1.DataCleaning-R/Data/RDS/CoachTeamMatchesBeforeWC.rds`
- `1.DataCleaning-R/Data/RDS/WC2026DistanceELO.rds`
- `1.DataCleaning-R/Data/CSV/WC2026_squads_source.html`

## Outputs

- `1.DataCleaning-R/Data/RDS/WC2026Experience.rds`
- `1.DataCleaning-R/Data/CSV/WC2026Experience.csv`

## Packages

In [1]:
library(worldcup)
library(tidyverse)
library(lubridate)
library(rvest)
library(here)

Warning message:
"package 'ggplot2' was built under R version 4.4.3"
Warning message:
"package 'purrr' was built under R version 4.4.3"
-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v dplyr     1.1.4     v readr     2.1.5
v forcats   1.0.0     v stringr   1.6.0
v ggplot2   4.0.1     v tibble    3.2.1
v lubridate 1.9.4     v tidyr     1.3.1
v purrr     1.2.1     
-- Conflicts ------------------------------------------ tidyverse_conflicts() --
x dplyr::filter() masks stats::filter()
x dplyr::lag()    masks stats::lag()
i Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: 'rvest'


The following object is masked from 'package:readr':

    guess_encoding


here() starts at /Users/eialnisman/Desktop/WC2026Forecast



## Base 2026 Features

Start from the 2026 team-level file created in this folder.

In [2]:
wc_2026_base <- readRDS(here("1.DataCleaning-R", "Data", "RDS", "WC2026DistanceELO.rds"))

if (!"tournament_id" %in% names(wc_2026_base)) {
    wc_2026_base <- wc_2026_base %>%
        mutate(tournament_id = "WC-2026", .before = team_name)
}

wc_2026_base %>%
    count()

n
<int>
48


## Helper Names

Use simple normalized names for matching 2026 squads and managers to the historical World Cup package data.

In [3]:
clean_name <- function(x) {
    x %>%
        str_remove_all("[[:space:]]*[(][^)]*[)]") %>%
        iconv(from = "UTF-8", to = "ASCII//TRANSLIT") %>%
        tolower() %>%
        str_replace_all("[^a-z ]", " ") %>%
        str_squish()
}

nft_to_full_name <- function(x) {
    if_else(
        str_detect(x, ","),
        str_squish(paste(
            str_trim(str_replace(x, "^[^,]+,", "")),
            str_trim(str_replace(x, ",.*$", ""))
        )),
        x
    )
}

wc_2026_team_name <- function(team) {
    case_when(
        team == "Cape Verde" ~ "Cabo Verde",
        team == "Cura\u00e7ao" ~ "Curacao",
        team == "Czech Republic" ~ "Czechia",
        team == "DR Congo" ~ "Congo DR",
        TRUE ~ team
    )
}

## Manager Experience

Summarize prior World Cup manager appearances before 2026 and match those records to the 2026 coach names.

In [4]:
manager_games <- worldcup::manager_appearances

manager_history_2026 <- manager_games %>%
    filter(tournament_id < "WC-2026") %>%
    mutate(
        manager_full_name = str_squish(paste(given_name, family_name)),
        coach_match_name = clean_name(manager_full_name)
    ) %>%
    group_by(coach_match_name) %>%
    summarize(
        prior_wc_matches_coached = n(),
        prior_world_cups_coached = n_distinct(tournament_id),
        matched_manager_name_worldcup = paste(unique(manager_full_name), collapse = "; "),
        .groups = "drop"
    )

coach_team_matches <- readRDS(here("1.DataCleaning-R", "Data", "RDS", "CoachTeamMatchesBeforeWC.rds"))

manager_experience_2026 <- coach_team_matches %>%
    filter(tournament_id == "WC-2026") %>%
    mutate(
        coach_full_name = nft_to_full_name(coach_name_nft),
        coach_match_name = clean_name(coach_full_name)
    ) %>%
    left_join(manager_history_2026, by = "coach_match_name") %>%
    mutate(
        prior_wc_matches_coached = replace_na(prior_wc_matches_coached, 0L),
        prior_world_cups_coached = replace_na(prior_world_cups_coached, 0L)
    ) %>%
    select(
        tournament_id,
        team_name,
        team_code,
        coach_name_nft,
        coach_full_name,
        matched_manager_name_worldcup,
        pre_wc_team_matches_coached = coach_team_matches_before_wc,
        prior_wc_matches_coached,
        prior_world_cups_coached
    )

manager_experience_2026 %>%
    arrange(desc(prior_world_cups_coached), team_name) %>%
    print(n = 10)

# A tibble: 48 x 9
   tournament_id team_name    team_code coach_name_nft           coach_full_name
   <chr>         <chr>        <chr>     <chr>                    <chr>          
 1 WC-2026       France       FRA       "Deschamps, Didier"      "Didier Descha~
 2 WC-2026       Croatia      HRV       "Dali\u0107, Zlatko"     "Zlatko Dali\u~
 3 WC-2026       Mexico       MEX       "Aguirre, Javier"        "Javier Aguirr~
 4 WC-2026       Portugal     PRT       "Mart\u00ednez, Roberto" "Roberto Mart\~
 5 WC-2026       Saudi Arabia SAU       "Renard, Herv\u00e9"     "Herv\u00e9 Re~
 6 WC-2026       Uruguay      URY       "Bielsa, Marcelo"        "Marcelo Biels~
 7 WC-2026       Algeria      DZA       "Petkovi\u0107, Vladimi~ "Vladimir Petk~
 8 WC-2026       Argentina    ARG       "Scaloni, Lionel"        "Lionel Scalon~
 9 WC-2026       Ghana        GHA       "Addo, Otto"             "Otto Addo"    
10 WC-2026       Iraq         IRQ       "Arnold, Graham"         "Graham Arnold"
# i 38 mo

## 2026 Squads

Read the 2026 squad tables and build the same kind of player roster structure used in the player experience notebook.

In [5]:
squad_url <- "https://episteme.tllm.fr/wiki/2026_FIFA_World_Cup_squads"
squad_cache <- here("1.DataCleaning-R", "Data", "CSV", "WC2026_squads_source.html")

squad_html <- if (file.exists(squad_cache)) {
    read_html(squad_cache)
} else {
    read_html(squad_url)
}

squad_nodes <- squad_html %>%
    html_elements("h2, h3, table")

current_team <- NA_character_
squad_tables <- list()

for (node in squad_nodes) {
    node_name <- xml2::xml_name(node)

    if (node_name == "h3") {
        current_team <- html_text2(node)
    }

    if (node_name == "table" && !is.na(current_team)) {
        table <- html_table(node, fill = TRUE)

        if (ncol(table) == 7 && "Player" %in% names(table)) {
            squad_tables[[length(squad_tables) + 1]] <- table %>%
                mutate(wiki_team_name = current_team)
        }

        current_team <- NA_character_
    }
}

squads_2026 <- bind_rows(squad_tables) %>%
    transmute(
        tournament_id = "WC-2026",
        team_name = wc_2026_team_name(wiki_team_name),
        player_name = Player,
        birth_date = ymd(str_extract(`Date of birth (age)`, "[0-9]{4}-[0-9]{2}-[0-9]{2}")),
        player_match_name = clean_name(Player)
    )

squads_2026 %>%
    count(team_name, name = "squad_players") %>%
    arrange(squad_players, team_name) %>%
    print(n = 10)

# A tibble: 48 x 2
   team_name              squad_players
   <chr>                          <int>
 1 Austria                           25
 2 Canada                            25
 3 Algeria                           26
 4 Argentina                         26
 5 Australia                         26
 6 Belgium                           26
 7 Bosnia and Herzegovina            26
 8 Brazil                            26
 9 Cabo Verde                        26
10 Colombia                          26
# i 38 more rows


## Player Experience

Use World Cup appearances before 2026 to count each squad player's prior World Cup experience, then aggregate to team level.

In [6]:
historical_player_names <- worldcup::squads %>%
    filter(tournament_id %in% c("WC-2010", "WC-2014", "WC-2018", "WC-2022")) %>%
    transmute(
        player_id,
        historical_player_name = str_squish(paste(given_name, family_name)),
        player_match_name = clean_name(historical_player_name)
    ) %>%
    distinct()

player_history_2026 <- worldcup::player_appearances %>%
    filter(tournament_id < "WC-2026") %>%
    group_by(player_id) %>%
    summarize(
        prior_wc_matches_played = n(),
        prior_world_cups_played = n_distinct(tournament_id),
        .groups = "drop"
    )

player_roster_experience_2026 <- squads_2026 %>%
    mutate(current_player_row = row_number()) %>%
    left_join(
        historical_player_names,
        by = "player_match_name",
        relationship = "many-to-many"
    ) %>%
    left_join(player_history_2026, by = "player_id") %>%
    mutate(
        prior_wc_matches_played = replace_na(prior_wc_matches_played, 0L),
        prior_world_cups_played = replace_na(prior_world_cups_played, 0L)
    ) %>%
    group_by(current_player_row, tournament_id, team_name, player_name, birth_date) %>%
    summarize(
        prior_wc_matches_played = max(prior_wc_matches_played),
        prior_world_cups_played = max(prior_world_cups_played),
        .groups = "drop"
    )

player_experience_2026 <- player_roster_experience_2026 %>%
    group_by(tournament_id, team_name) %>%
    summarize(
        squad_players = n_distinct(current_player_row),
        avg_age = mean(as.numeric(as.Date("2026-06-11") - birth_date) / 365.25, na.rm = TRUE),
        missing_player_birth_dates = sum(is.na(birth_date)),
        players_with_prior_wc = sum(prior_wc_matches_played > 0),
        players_with_multiple_prior_wcs = sum(prior_world_cups_played > 1),
        .groups = "drop"
    )

player_experience_2026 %>%
    arrange(desc(players_with_multiple_prior_wcs), team_name) %>%
    print(n = 10)

# A tibble: 48 x 7
   tournament_id team_name   squad_players avg_age missing_player_birth_dates
   <chr>         <chr>               <int>   <dbl>                      <int>
 1 WC-2026       Iran                   26    30.3                          0
 2 WC-2026       Belgium                26    27.6                          0
 3 WC-2026       Colombia               26    30.1                          0
 4 WC-2026       England                26    27.1                          0
 5 WC-2026       Australia              26    27.4                          0
 6 WC-2026       Croatia                26    28.3                          0
 7 WC-2026       Germany                26    28.0                          0
 8 WC-2026       Mexico                 26    27.9                          0
 9 WC-2026       Switzerland            26    28.3                          0
10 WC-2026       Uruguay                26    28.7                          0
# i 38 more rows
# i 2 more variables: player

## Combine Experience

Join the manager and player experience features onto the 2026 distance and ELO data.

In [7]:
wc_2026_experience <- wc_2026_base %>%
    left_join(
        manager_experience_2026 %>%
            select(team_name, coach_name_nft, coach_full_name, pre_wc_team_matches_coached, prior_world_cups_coached),
        by = "team_name"
    ) %>%
    left_join(
        player_experience_2026 %>%
            select(team_name, squad_players, avg_age, missing_player_birth_dates, players_with_prior_wc, players_with_multiple_prior_wcs),
        by = "team_name"
    ) %>%
    arrange(team_name)

wc_2026_experience %>%
    summarize(
        teams = n(),
        missing_prior_world_cups_coached = sum(is.na(prior_world_cups_coached)),
        missing_players_with_multiple_prior_wcs = sum(is.na(players_with_multiple_prior_wcs)),
        missing_avg_age = sum(is.na(avg_age)),
        .groups = "drop"
    )

teams,missing_prior_world_cups_coached,missing_players_with_multiple_prior_wcs,missing_avg_age
<int>,<int>,<int>,<int>
48,0,0,0


Inspect

In [8]:
wc_2026_experience

tournament_id,team_name,team_id,team_code,distance_from_host_km,elo_2026,coach_name_nft,coach_full_name,pre_wc_team_matches_coached,prior_world_cups_coached,squad_players,avg_age,missing_player_birth_dates,players_with_prior_wc,players_with_multiple_prior_wcs
<chr>,<chr>,<chr>,<chr>,<dbl>,<int>,<chr>,<chr>,<int>,<int>,<int>,<dbl>,<int>,<int>,<int>
WC-2026,Algeria,T-01,DZA,8467.886,1757,"Petkovi<U+0107>, Vladimir",Vladimir Petkovi<U+0107>,27,1,26,26.85410,0,3,0
WC-2026,Argentina,T-03,ARG,9076.351,2113,"Scaloni, Lionel",Lionel Scaloni,94,1,26,29.05228,0,16,3
WC-2026,Australia,T-04,AUS,14653.989,1774,"Popovi<U+0107>, Tony",Tony Popovi<U+0107>,16,0,26,27.35945,0,8,4
WC-2026,Austria,T-05,AUT,8351.637,1818,"Rangnick, Ralf",Ralf Rangnick,44,0,25,28.60901,0,0,0
WC-2026,Belgium,T-06,BEL,7608.706,1850,"Garcia, Rudi",Rudi Garcia,12,0,26,27.61449,0,11,6
WC-2026,Bosnia and Herzegovina,T-08,BIH,8799.915,1572,"Barbarez, Sergej",Sergej Barbarez,20,0,26,26.42247,0,2,0
WC-2026,Brazil,T-09,BRA,7396.652,1978,"Ancelotti, Carlo",Carlo Ancelotti,10,0,26,29.20276,0,5,0
WC-2026,Cabo Verde,NA,CPV,7732.128,1561,"Bubista,",Bubista,59,0,26,29.67641,0,0,0
WC-2026,Canada,T-12,CAN,0.000,1802,"Marsch, Jesse",Jesse Marsch,30,0,25,27.10330,0,10,0


In [11]:
wc_2026_experience <- wc_2026_experience %>%
    select(team_name,team_id,team_code,prior_world_cups_coached,avg_age, players_with_multiple_prior_wcs)

## Save

Save RDS

In [12]:
saveRDS(wc_2026_experience, here("1.DataCleaning-R", "Data", "RDS", "WC2026Experience.rds"))